# Traffic Violation Model Training — Kaggle
### Trains the 4-class violation detector: `Plate, WithHelmet, WithoutHelmet, TripleRiding`

**How to use this on Kaggle:**
1. Create a new Kaggle Notebook.
2. Add this dataset via **+ Add Input**: `devgurucodes/trafffic-violations-triple-riding-no-helmet-plate`
   (this is the dataset your `data.yaml` was pointing at — confirm the exact slug in Kaggle's search
   if it's moved, since dataset slugs occasionally change).
3. **Settings → Accelerator → GPU T4 x2** (or whatever GPU option is available to you). Training
   on CPU will be extremely slow — don't skip this.
4. **Settings → Internet → On.** Required for `pip install` and to auto-download `yolo26n.pt`.
5. Paste these cells in, run top to bottom.
6. After training, run the **export cell** at the end — it zips your weights into
   `/kaggle/working/` so you can download them from the Kaggle output panel.

**This is configured as a fast sanity run (~20-30 min on a T4), not a full 100-epoch run.**
The goal first is to confirm the whole chain works — dataset paths resolve, training runs,
weights save, classes are correct — before you spend hours on a longer run. Once this completes
cleanly, bump `EPOCHS` up (see Cell 3) and re-run for a real submission-quality model.


In [ ]:
# Cell 1: Install / upgrade ultralytics
# Kaggle images usually have an older ultralytics preinstalled — force latest.

!pip install -q -U ultralytics


In [ ]:
# Cell 2: Sanity-check the environment before burning GPU time on a bad config

import torch
import os
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU before continuing,")
    print("otherwise this will train on CPU and take hours instead of minutes.")

# ---------------------------------------------------------------
# Locate the dataset. Kaggle mounts added datasets under /kaggle/input/<dataset-slug>/...
# Adjust DATASET_SLUG if your dataset shows up under a different folder name in
# /kaggle/input — print the listing below to check before assuming the path.
# ---------------------------------------------------------------
print("\nContents of /kaggle/input:")
for p in Path("/kaggle/input").iterdir():
    print(" -", p)


In [ ]:
# Cell 3: Config

from pathlib import Path

# ---------------------------------------------------------------
# Find data.yaml automatically instead of hardcoding a brittle path —
# Kaggle dataset folder structures sometimes nest one level deeper than expected.
# ---------------------------------------------------------------
candidates = list(Path("/kaggle/input").rglob("data.yaml"))
print("Found data.yaml candidate(s):")
for c in candidates:
    print(" -", c)

if not candidates:
    raise FileNotFoundError(
        "No data.yaml found under /kaggle/input. Make sure the dataset is added via "
        "'+ Add Input' in the notebook sidebar before running this cell."
    )

# If multiple data.yaml files are found (e.g. nested train/valid/test copies), pick the
# one at the shallowest path — that's almost always the root config.
DATA_YAML_PATH = str(min(candidates, key=lambda p: len(p.parts)))
print("\nUsing data.yaml at:", DATA_YAML_PATH)

with open(DATA_YAML_PATH) as f:
    print("\n--- data.yaml contents ---")
    print(f.read())

# ---------------------------------------------------------------
# Training config — FAST SANITY RUN (~20-30 min on a Kaggle T4)
# ---------------------------------------------------------------
BASE_MODEL = "yolo26n.pt"   # nano model, auto-downloads on first use (needs Internet ON)
EPOCHS = 15                  # bump to 100 for a full quality run once this works end-to-end
IMG_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0                   # GPU index; use "cpu" only if no GPU is available
WORKERS = 4

print(f"\nTraining config: model={BASE_MODEL}, epochs={EPOCHS}, imgsz={IMG_SIZE}, "
      f"batch={BATCH_SIZE}, device={DEVICE}")


In [ ]:
# Cell 4: Train

from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    hsv_h=0.1,   # kept from your original config (hue augmentation)
    project="/kaggle/working/runs",
    name="violation_detector",
    exist_ok=True,
)

print("\nTraining complete.")
print("Best weights:", model.trainer.best)
print("Last weights:", model.trainer.last)


In [ ]:
# Cell 5: Quick validation on the held-out val split, so you have real numbers
# (mAP, precision, recall) before you trust this model in the pipeline.

from ultralytics import YOLO

best_weights_path = str(model.trainer.best)
trained_model = YOLO(best_weights_path)

metrics = trained_model.val(data=DATA_YAML_PATH)

print("\n--- Validation summary ---")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Per-class mAP50:")
for idx, name in trained_model.names.items():
    try:
        print(f"  {name}: {metrics.box.ap50[idx]:.3f}")
    except (IndexError, TypeError):
        pass


In [ ]:
# Cell 6: Run inference on a few validation images and display them, so you can
# eyeball whether the model is actually detecting the right things before downloading.

import matplotlib.pyplot as plt
from PIL import Image
import yaml

with open(DATA_YAML_PATH) as f:
    data_cfg = yaml.safe_load(f)

# val path in data.yaml is relative to the data.yaml file's directory
val_images_dir = Path(DATA_YAML_PATH).parent / data_cfg["val"]
sample_images = list(val_images_dir.glob("*.jpg"))[:6] + list(val_images_dir.glob("*.png"))[:6]
sample_images = sample_images[:6]

if not sample_images:
    print(f"No sample images found in {val_images_dir} — skipping visual check.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flatten(), sample_images):
        pred = trained_model.predict(str(img_path), conf=0.25, verbose=False)
        annotated = pred[0].plot()  # BGR numpy array with boxes drawn
        ax.imshow(annotated[:, :, ::-1])  # BGR -> RGB for matplotlib
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/sanity_check_predictions.png", dpi=100)
    plt.show()
    print("Saved grid to /kaggle/working/sanity_check_predictions.png")


In [ ]:
# Cell 7: Package the trained weights for download
# Run this last. It copies best.pt to a predictable location and zips the full
# run folder (weights + training curves + confusion matrix) so you can grab
# everything from the Kaggle "Output" panel on the right side of the notebook.

import shutil

EXPORT_DIR = Path("/kaggle/working/export")
EXPORT_DIR.mkdir(exist_ok=True)

# Copy best.pt to a simple, predictable filename
shutil.copy(model.trainer.best, EXPORT_DIR / "best.pt")
shutil.copy(model.trainer.last, EXPORT_DIR / "last.pt")
shutil.copy(DATA_YAML_PATH, EXPORT_DIR / "data.yaml")

print("Exported to:", EXPORT_DIR)
print("Files:")
for f in EXPORT_DIR.iterdir():
    print(" -", f, f"({f.stat().st_size / 1e6:.1f} MB)")

# Zip the whole training run (weights, results.png, confusion_matrix.png, args.yaml, etc.)
run_dir = Path(model.trainer.save_dir)
zip_path = "/kaggle/working/violation_detector_run"
shutil.make_archive(zip_path, "zip", run_dir)
print(f"\nFull run zipped to: {zip_path}.zip")
print("\nNext step: go to the 'Output' tab in the Kaggle notebook sidebar, find")
print("'export/best.pt' (or the zip), and download it. Then point your inference")
print("notebook's VIOLATION_MODEL_PATH at this downloaded best.pt file.")
